# Start here: shared IPI design and repeatable live tests

Use this notebook to build the repository's **hw/sys.bd ? XSA ? mb BSP ? bench.elf ? demo.pdi** image, then program it and compare new hardware and Python timings inline. The AXI Timer is part of this BD. The image embeds the ELF in BRAM, so **no separate ELF download is required**.

Copy config.example.json to config.json and configure your tool root, short local build directory, exact cable and UART, matching hardware Vivado/server, recovery PDI and SHA-256. The build requires 2026.1; the hardware client must match your server. The local config is ignored by Git.

Close other COM-port terminals and active debug sessions before running LIVE=True. The runtime runner programs only the selected SCU35 and reloads your configured recovery image afterward. It performs no flash or eFUSE writes. Set LIVE=True explicitly for every fresh test; otherwise the output clearly identifies recorded board evidence plus fresh Python timing.

In [ ]:
from pathlib import Path
import sys, os, json, time, hashlib
import pandas as pd
ROOT = next((p for base in [Path.cwd(), *Path.cwd().parents]
             for p in (base, base/'scu35') if (p/'lab.py').is_file()), None)
assert ROOT is not None, 'Open the notebook from the repository checkout'
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT/'host'))
from lab import config, tool
from notebook import runtime, show_runtime, LiveRows, command, hardware, check_build
c = config()
OUT = Path(c['build_dir']).resolve()
print('Python:', sys.executable)
print('Build:', OUT)


In [ ]:
print(json.dumps(c, indent=2))
BUILD_PHASES = []  # Fresh build: ['hardware','platform','firmware','image']
# Use a FRESH short build directory for a new hardware/platform build.
if BUILD_PHASES:
    command('doctor', c=c)
    for phase in BUILD_PHASES:
        command('build','--phase',phase,c=c)
else:
    print('Build skipped; reuse your configured existing repository IPI build.')


In [ ]:
INSPECT_BUILD = False
if INSPECT_BUILD:
    pdi = check_build(c)
    command('inspect','--pdi',pdi,'--elf',OUT/'plm.elf',c=c)
    print('Implementation:', (OUT/'implementation_checks.json').read_text())


## Fresh hardware and Python run
The full 31-row suite runs once per invocation, with provisional UART timing updates followed by strict functional validation. Set REPEATS above one for independent hardware/Python benchmark runs. No previous successful capture is substituted after a live failure. The final report has new timestamps and exact capture/PDI hashes.

In [ ]:
LIVE = False
REPEATS = 1
reports = runtime(live=LIVE, repeats=REPEATS, c=c,
                  on_line=LiveRows() if LIVE else None)
comparison = show_runtime(reports)


Continue with notebooks 01?05 to rerun and explain individual feature groups; each can independently invoke this same hardware/Python flow. Notebook 08 benchmarks the shared IPI runtime, 06?07 handle image signing/encryption, and 09 inspects current PDI/PLM files. Device operation timing excludes host programming, transport and recovery. PUF has no physical Python equivalent; boot counters are not instrumented; eFUSE provisioning is excluded.